# Parquet Features Validation & Testing

This notebook tests and validates the parquet files generated by the ingestor persistence pipeline.


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from datetime import datetime
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)


## 1. Discover and Load Parquet Files


In [ ]:
# Find all parquet files in data/features
features_dir = Path("../data/features")
parquet_files = sorted(features_dir.glob("features_*.parquet"))

print(f"Found {len(parquet_files)} parquet file(s)")
for f in parquet_files[:5]:  # Show first 5
    print(f"  - {f.name}")


## 2. Test: Load Single Parquet File


In [ ]:
def test_load_single_parquet(filepath: Path) -> pl.DataFrame:
    """Test loading a single parquet file."""
    assert filepath.exists(), f"File not found: {filepath}"
    
    df = pl.read_parquet(filepath)
    assert df.height > 0, "DataFrame is empty"
    
    print(f"✓ Loaded {filepath.name}")
    print(f"  Rows: {df.height}, Columns: {df.width}")
    print(f"  Columns: {df.columns[:10]}..." if len(df.columns) > 10 else f"  Columns: {df.columns}")
    
    return df

if parquet_files:
    df_single = test_load_single_parquet(parquet_files[0])
    print("\nFirst 3 rows:")
    print(df_single.head(3))


## 3. Test: Schema Validation


In [ ]:
def test_schema_validation(df: pl.DataFrame) -> bool:
    """Test that required columns exist and have correct types."""
    required_cols = [
        "timestamp", "best_bid", "best_ask", "mid_price", "spread",
        "order_flow_pressure", "signed_count_momentum"
    ]
    
    missing = [col for col in required_cols if col not in df.columns]
    assert len(missing) == 0, f"Missing required columns: {missing}"
    
    # Check timestamp is string
    assert df["timestamp"].dtype == pl.Utf8, "timestamp should be string"
    
    print("✓ Schema validation passed")
    print(f"  Total columns: {df.width}")
    print(f"  Required columns present: {len(required_cols)}")
    
    return True

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    test_schema_validation(df)


## 4. Test: Data Quality Checks


In [ ]:
def test_data_quality(df: pl.DataFrame) -> bool:
    """Test data quality: no nulls in critical fields, valid ranges."""
    # Convert to pandas for easier null checking
    df_pd = df.to_pandas()
    
    # Check timestamp parsing
    timestamps = pd.to_datetime(df_pd["timestamp"], errors='coerce')
    assert timestamps.notna().all(), "All timestamps must be valid"
    
    # Check numeric columns for reasonable ranges
    if "mid_price" in df_pd.columns:
        mid_prices = df_pd["mid_price"].dropna()
        assert (mid_prices > 0).all(), "mid_price must be positive"
    
    if "spread" in df_pd.columns:
        spreads = df_pd["spread"].dropna()
        assert (spreads >= 0).all(), "spread must be non-negative"
    
    print("✓ Data quality checks passed")
    print(f"  Valid timestamps: {timestamps.notna().sum()}/{len(timestamps)}")
    
    return True

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    test_data_quality(df)


## 5. Test: Load Multiple Files (Concatenation)


In [ ]:
def test_load_multiple_files(filepaths: list[Path], limit: int = None) -> pl.DataFrame:
    """Test loading and concatenating multiple parquet files."""
    if limit:
        filepaths = filepaths[:limit]
    
    dfs = [pl.read_parquet(fp) for fp in filepaths]
    
    # Ensure all have same schema
    first_schema = dfs[0].schema
    for i, df in enumerate(dfs[1:], 1):
        assert df.schema == first_schema, f"Schema mismatch in file {i+1}"
    
    combined = pl.concat(dfs)
    
    print(f"✓ Loaded and concatenated {len(filepaths)} files")
    print(f"  Total rows: {combined.height}")
    print(f"  Columns: {combined.width}")
    
    return combined

if len(parquet_files) > 1:
    df_combined = test_load_multiple_files(parquet_files, limit=5)
else:
    print("Only one parquet file found, skipping concatenation test")


## 6. Test: JSON Column Parsing (top_bids, top_asks)


In [ ]:
def test_json_columns(df: pl.DataFrame) -> bool:
    """Test that JSON columns (top_bids, top_asks) can be parsed."""
    df_pd = df.to_pandas()
    
    if "top_bids" in df_pd.columns:
        sample = df_pd["top_bids"].dropna().iloc[0] if df_pd["top_bids"].notna().any() else None
        if sample:
            parsed = json.loads(sample)
            assert isinstance(parsed, list), "top_bids should be a list"
            print(f"✓ top_bids parsed successfully: {len(parsed)} entries")
    
    if "top_asks" in df_pd.columns:
        sample = df_pd["top_asks"].dropna().iloc[0] if df_pd["top_asks"].notna().any() else None
        if sample:
            parsed = json.loads(sample)
            assert isinstance(parsed, list), "top_asks should be a list"
            print(f"✓ top_asks parsed successfully: {len(parsed)} entries")
    
    return True

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    test_json_columns(df)


## 7. Visualization Tests


In [ ]:
def plot_mid_price_over_time(df: pl.DataFrame):
    """Plot mid_price over time."""
    df_pd = df.to_pandas()
    df_pd["ts"] = pd.to_datetime(df_pd["timestamp"])
    
    if "mid_price" in df_pd.columns:
        plt.figure(figsize=(14, 6))
        plt.plot(df_pd["ts"], df_pd["mid_price"], alpha=0.7, linewidth=1)
        plt.xlabel("Time")
        plt.ylabel("Mid Price")
        plt.title("Mid Price Over Time")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        print("✓ Mid price plot generated")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_mid_price_over_time(df)


## 8. Comprehensive Feature Visualizations


In [ ]:
def plot_all_orderbook_metrics(df: pl.DataFrame):
    """Plot all orderbook-related metrics."""
    df_pd = df.to_pandas()
    df_pd["ts"] = pd.to_datetime(df_pd["timestamp"])
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle("Orderbook Metrics Over Time", fontsize=16)
    
    # Mid price
    if "mid_price" in df_pd.columns:
        axes[0, 0].plot(df_pd["ts"], df_pd["mid_price"], alpha=0.7, linewidth=1)
        axes[0, 0].set_title("Mid Price")
        axes[0, 0].set_ylabel("Price")
        axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Spread
    if "spread" in df_pd.columns:
        axes[0, 1].plot(df_pd["ts"], df_pd["spread"], alpha=0.7, linewidth=1, color='orange')
        axes[0, 1].set_title("Spread")
        axes[0, 1].set_ylabel("Spread")
        axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Imbalance
    if "imbalance" in df_pd.columns:
        axes[1, 0].plot(df_pd["ts"], df_pd["imbalance"], alpha=0.7, linewidth=1, color='green')
        axes[1, 0].set_title("Order Book Imbalance")
        axes[1, 0].set_ylabel("Imbalance")
        axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Order flow pressure
    if "order_flow_pressure" in df_pd.columns:
        axes[1, 1].plot(df_pd["ts"], df_pd["order_flow_pressure"], alpha=0.7, linewidth=1, color='red')
        axes[1, 1].set_title("Order Flow Pressure")
        axes[1, 1].set_ylabel("Pressure")
        axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Order flow imbalance
    if "order_flow_imbalance" in df_pd.columns:
        axes[2, 0].plot(df_pd["ts"], df_pd["order_flow_imbalance"], alpha=0.7, linewidth=1, color='purple')
        axes[2, 0].set_title("Order Flow Imbalance")
        axes[2, 0].set_ylabel("Imbalance")
        axes[2, 0].tick_params(axis='x', rotation=45)
    
    # Microprice
    if "microprice" in df_pd.columns:
        axes[2, 1].plot(df_pd["ts"], df_pd["microprice"], alpha=0.7, linewidth=1, color='brown')
        axes[2, 1].set_title("Microprice")
        axes[2, 1].set_ylabel("Price")
        axes[2, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    print("✓ Orderbook metrics plot generated")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_all_orderbook_metrics(df)


In [ ]:
def plot_trade_metrics(df: pl.DataFrame):
    """Plot all trade-related metrics."""
    df_pd = df.to_pandas()
    df_pd["ts"] = pd.to_datetime(df_pd["timestamp"])
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle("Trade Metrics Over Time", fontsize=16)
    
    # VWAP total
    if "vwap_total" in df_pd.columns:
        axes[0, 0].plot(df_pd["ts"], df_pd["vwap_total"], alpha=0.7, linewidth=1)
        axes[0, 0].set_title("VWAP Total")
        axes[0, 0].set_ylabel("Price")
        axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Trade imbalance
    if "trade_imbalance" in df_pd.columns:
        axes[0, 1].plot(df_pd["ts"], df_pd["trade_imbalance"], alpha=0.7, linewidth=1, color='orange')
        axes[0, 1].set_title("Trade Imbalance")
        axes[0, 1].set_ylabel("Imbalance")
        axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Signed count momentum
    if "signed_count_momentum" in df_pd.columns:
        axes[1, 0].plot(df_pd["ts"], df_pd["signed_count_momentum"], alpha=0.7, linewidth=1, color='green')
        axes[1, 0].set_title("Signed Count Momentum")
        axes[1, 0].set_ylabel("Momentum")
        axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Trade rate
    if "trade_rate_10s" in df_pd.columns:
        axes[1, 1].plot(df_pd["ts"], df_pd["trade_rate_10s"], alpha=0.7, linewidth=1, color='red')
        axes[1, 1].set_title("Trade Rate (10s)")
        axes[1, 1].set_ylabel("Trades/sec")
        axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Aggressor ratio 10
    if "aggr_ratio_10" in df_pd.columns:
        axes[2, 0].plot(df_pd["ts"], df_pd["aggr_ratio_10"], alpha=0.7, linewidth=1, color='purple')
        axes[2, 0].set_title("Aggressor Ratio (10)")
        axes[2, 0].set_ylabel("Ratio")
        axes[2, 0].tick_params(axis='x', rotation=45)
    
    # Price change
    if "price_change" in df_pd.columns:
        axes[2, 1].plot(df_pd["ts"], df_pd["price_change"], alpha=0.7, linewidth=1, color='brown')
        axes[2, 1].set_title("Price Change")
        axes[2, 1].set_ylabel("Change")
        axes[2, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    print("✓ Trade metrics plot generated")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_trade_metrics(df)


In [ ]:
def plot_illiquidity_metrics(df: pl.DataFrame):
    """Plot all illiquidity metrics."""
    df_pd = df.to_pandas()
    df_pd["ts"] = pd.to_datetime(df_pd["timestamp"])
    
    illiquidity_cols = ["roll_spread", "amihuds_lambda", "kyles_lambda", "hasbroucks_lambda", "vpin"]
    available_cols = [col for col in illiquidity_cols if col in df_pd.columns and df_pd[col].notna().any()]
    
    if not available_cols:
        print("⚠️ No illiquidity metrics found in data")
        return
    
    n_cols = len(available_cols)
    n_rows = (n_cols + 1) // 2
    
    fig, axes = plt.subplots(n_rows, 2, figsize=(16, 4 * n_rows))
    fig.suptitle("Illiquidity Metrics Over Time", fontsize=16)
    
    axes = axes.flatten() if n_cols > 1 else [axes] if n_rows == 1 else axes
    
    for idx, col in enumerate(available_cols):
        ax = axes[idx] if n_cols > 1 else axes
        ax.plot(df_pd["ts"], df_pd[col], alpha=0.7, linewidth=1)
        ax.set_title(col.replace("_", " ").title())
        ax.set_ylabel("Value")
        ax.tick_params(axis='x', rotation=45)
    
    # Hide unused subplots
    for idx in range(len(available_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    print(f"✓ Illiquidity metrics plot generated ({len(available_cols)} metrics)")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_illiquidity_metrics(df)


In [ ]:
def plot_entropy_metrics(df: pl.DataFrame):
    """Plot all entropy metrics."""
    df_pd = df.to_pandas()
    df_pd["ts"] = pd.to_datetime(df_pd["timestamp"])
    
    # Tick entropy columns
    tick_entropy_cols = [col for col in df_pd.columns if col.startswith("tick_entropy_")]
    volume_entropy_cols = [col for col in df_pd.columns if col.startswith("volume_tick_entropy_")]
    
    if not tick_entropy_cols and not volume_entropy_cols:
        print("⚠️ No entropy metrics found in data")
        return
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    fig.suptitle("Entropy Metrics Over Time", fontsize=16)
    
    # Plot tick entropy
    if tick_entropy_cols:
        for col in tick_entropy_cols:
            if df_pd[col].notna().any():
                axes[0].plot(df_pd["ts"], df_pd[col], alpha=0.7, linewidth=1, label=col.replace("tick_entropy_", ""))
        axes[0].set_title("Tick Entropy (Multiple Windows)")
        axes[0].set_ylabel("Entropy")
        axes[0].legend()
        axes[0].tick_params(axis='x', rotation=45)
    
    # Plot volume tick entropy
    if volume_entropy_cols:
        for col in volume_entropy_cols:
            if df_pd[col].notna().any():
                axes[1].plot(df_pd["ts"], df_pd[col], alpha=0.7, linewidth=1, label=col.replace("volume_tick_entropy_", ""))
        axes[1].set_title("Volume Tick Entropy (Multiple Windows)")
        axes[1].set_ylabel("Entropy")
        axes[1].set_xlabel("Time")
        axes[1].legend()
        axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    print(f"✓ Entropy metrics plot generated (tick: {len(tick_entropy_cols)}, volume: {len(volume_entropy_cols)})")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_entropy_metrics(df)


In [ ]:
def plot_correlation_heatmap(df: pl.DataFrame):
    """Plot correlation heatmap of key numeric features."""
    df_pd = df.to_pandas()
    
    # Select key numeric columns
    key_cols = [
        "mid_price", "spread", "imbalance", "order_flow_pressure", "order_flow_imbalance",
        "trade_imbalance", "signed_count_momentum", "trade_rate_10s",
        "roll_spread", "amihuds_lambda", "kyles_lambda", "vpin",
        "tick_entropy_10s", "volume_tick_entropy_10s"
    ]
    
    available_cols = [col for col in key_cols if col in df_pd.columns]
    numeric_df = df_pd[available_cols].select_dtypes(include=[np.number])
    
    if numeric_df.empty:
        print("⚠️ No numeric columns available for correlation")
        return
    
    corr = numeric_df.corr()
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title("Feature Correlation Heatmap", fontsize=16)
    plt.tight_layout()
    plt.show()
    print("✓ Correlation heatmap generated")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_correlation_heatmap(df)


In [ ]:
def plot_distributions(df: pl.DataFrame):
    """Plot distributions of key metrics."""
    df_pd = df.to_pandas()
    
    key_cols = [
        "spread", "imbalance", "order_flow_pressure", "trade_imbalance",
        "roll_spread", "vpin", "tick_entropy_10s"
    ]
    
    available_cols = [col for col in key_cols if col in df_pd.columns and df_pd[col].notna().any()]
    
    if not available_cols:
        print("⚠️ No columns available for distribution plots")
        return
    
    n_cols = len(available_cols)
    n_rows = (n_cols + 2) // 3
    
    fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5 * n_rows))
    fig.suptitle("Distribution of Key Metrics", fontsize=16)
    
    axes = axes.flatten() if n_cols > 1 else [axes] if n_rows == 1 else axes
    
    for idx, col in enumerate(available_cols):
        ax = axes[idx]
        data = df_pd[col].dropna()
        if len(data) > 0:
            ax.hist(data, bins=50, edgecolor='black', alpha=0.7)
            ax.set_title(col.replace("_", " ").title())
            ax.set_xlabel("Value")
            ax.set_ylabel("Frequency")
    
    # Hide unused subplots
    for idx in range(len(available_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    print(f"✓ Distribution plots generated ({len(available_cols)} metrics)")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_distributions(df)


## 9. Summary Statistics


## 10. Run All Tests


## 11. Generate All Visualizations at Once


In [ ]:
def generate_all_visualizations(filepaths: list[Path], limit: int = None):
    """Generate all visualizations for the parquet files."""
    if not filepaths:
        print("❌ No parquet files found")
        return
    
    if limit:
        filepaths = filepaths[:limit]
    
    # Load and combine files
    dfs = [pl.read_parquet(fp) for fp in filepaths]
    df = pl.concat(dfs) if len(dfs) > 1 else dfs[0]
    
    print(f"Generating visualizations for {df.height} rows from {len(filepaths)} file(s)\n")
    
    # Generate all plots
    plot_mid_price_over_time(df)
    print()
    
    plot_order_flow_pressure(df)
    print()
    
    plot_all_orderbook_metrics(df)
    print()
    
    plot_trade_metrics(df)
    print()
    
    plot_illiquidity_metrics(df)
    print()
    
    plot_entropy_metrics(df)
    print()
    
    plot_correlation_heatmap(df)
    print()
    
    plot_distributions(df)
    print()
    
    print("✅ All visualizations generated!")

# Generate all visualizations
if parquet_files:
    generate_all_visualizations(parquet_files, limit=3)
else:
    print("⚠️ No parquet files found. Run the ingestor first to generate data.")


In [ ]:
def plot_order_flow_pressure(df: pl.DataFrame):
    """Plot histogram of order_flow_pressure."""
    df_pd = df.to_pandas()
    
    if "order_flow_pressure" in df_pd.columns:
        plt.figure(figsize=(10, 6))
        df_pd["order_flow_pressure"].hist(bins=50, edgecolor='black')
        plt.xlabel("Order Flow Pressure")
        plt.ylabel("Frequency")
        plt.title("Distribution of Order Flow Pressure")
        plt.tight_layout()
        plt.show()
        print("✓ Order flow pressure histogram generated")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    plot_order_flow_pressure(df)


## 8. Summary Statistics


In [ ]:
def print_summary_stats(df: pl.DataFrame):
    """Print summary statistics for numeric columns."""
    numeric_cols = [col for col, dtype in df.schema.items() 
                    if dtype in [pl.Float64, pl.Int64, pl.Float32, pl.Int32]]
    
    if numeric_cols:
        print("Summary Statistics:")
        print(df.select(numeric_cols[:10]).describe())  # Limit to first 10
    else:
        print("No numeric columns found")

if parquet_files:
    df = pl.read_parquet(parquet_files[0])
    print_summary_stats(df)


## 9. Run All Tests


In [ ]:
def run_all_tests(filepaths: list[Path]):
    """Run all validation tests."""
    if not filepaths:
        print("❌ No parquet files found")
        return False
    
    print(f"Running tests on {len(filepaths)} file(s)\n")
    
    try:
        # Test 1: Load single file
        df = test_load_single_parquet(filepaths[0])
        print()
        
        # Test 2: Schema validation
        test_schema_validation(df)
        print()
        
        # Test 3: Data quality
        test_data_quality(df)
        print()
        
        # Test 4: JSON columns
        test_json_columns(df)
        print()
        
        # Test 5: Multiple files (if available)
        if len(filepaths) > 1:
            test_load_multiple_files(filepaths, limit=3)
        
        print("\n✅ All tests passed!")
        return True
        
    except AssertionError as e:
        print(f"\n❌ Test failed: {e}")
        return False
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        return False

# Run all tests
run_all_tests(parquet_files)
